In [ ]:
import numpy as np
import torch
from pathlib import Path
import problem_config as cfg
from pathlib import Path
import matplotlib.pyplot as plt
import pickle


def kl_gaussian(m0, P0, m1, P1, jitter=1e-9):
    """
    KL( N(m0,P0) || N(m1,P1) ) per step.
    Uses stable solves and adds jitter to keep PD.
    """
    n = P0.shape[-1]
    # jitter for stability
    P0j = P0.copy()
    P1j = P1.copy()
    for M in (P0j, P1j):
        M[...] = 0.5 * (M + M.transpose(0,1))  # symmetrize (if batched later)
    P0j = P0j + jitter * np.eye(n)
    P1j = P1j + jitter * np.eye(n)

    # Cholesky of P1 to get logdet and solve P1^{-1} * X
    L1 = np.linalg.cholesky(P1j)
    logdet_P1 = 2.0 * np.sum(np.log(np.diag(L1)))
    # logdet of P0
    L0 = np.linalg.cholesky(P0j)
    logdet_P0 = 2.0 * np.sum(np.log(np.diag(L0)))

    # trace(P1^{-1} P0) via solves
    # solve L1 * Y = P0  => Y, then solve L1^T * Z = Y => Z = P1^{-1} P0
    Y = np.linalg.solve(L1, P0j)
    Z = np.linalg.solve(L1.T, Y)
    tr_term = np.trace(Z)

    d = (m1 - m0).reshape(-1, 1)  # (n,1)
    # quadratic form d^T P1^{-1} d via solves
    y = np.linalg.solve(L1, d)
    q = y.T @ y

    return float(np.asarray(0.5 * (logdet_P1 - logdet_P0 - n + tr_term + q)).reshape(()))


In [ ]:

base_dir = Path(".").resolve()
result_dir = base_dir / "result"
data_dir = base_dir / "problem_data"


# ground-truth state path
# --- Paths relative to problem_config.py ---
y = np.load(data_dir / "obs_path.npy")          # (T+1, m)
state_true = np.load(data_dir / "state_path.npy")   # (T_true, n_true)


# load results
kf = np.load(result_dir / "kf.npz") # ground truth
ms_kf = kf["posterior_mean"]         # (T, n)
Ps_kf = kf["posterior_cov"]          # (T, n, n)
print(f"KF:   {ms_kf.shape}, {Ps_kf.shape}")
T_total, n = ms_kf.shape
t = np.arange(T_total)

# load benchmark result
method_list = ["pf", "enkf", "ensf"]
load_result_all = {}
for name in method_list:
    with open(result_dir / f"{name}.pkl", "rb") as f:
        load_result_all[name] = pickle.load(f)



# bench mark
with open("result/iensf.pkl", "rb") as f:
    iensf_results = pickle.load(f)

In [ ]:
# get all kl
kl_all_method = {}
for key, loaded_results in load_result_all.items():
    kl_rep_all = []
    n_rep = len(loaded_results)
    for rep_id, (ms, Ps) in enumerate(loaded_results):
        # compute KL per step
        kl_all = np.zeros(T_total)
        for k in range(T_total):
            m0, P0 = ms_kf[k], Ps_kf[k]
            m, P = ms[k], Ps[k]
            kl_all[k] = kl_gaussian(m0, P0, m, P)
        kl_rep_all.append(kl_all)
    kl_all_method[key] = np.array(kl_rep_all)
    

    
# iensf
N_iter_vec = [1,2,3,4,5]
iensf_kl_rep = {}
for rep_id in range(n_rep):
    for N_iter in N_iter_vec:
        result_temp = iensf_results[f'iensf_{N_iter}_rep{rep_id}']
        ms, Ps = result_temp
        # compute KL per step
        kl_all = np.zeros(T_total)
        for k in range(T_total):
            m0, P0 = ms_kf[k], Ps_kf[k]
            m, P = ms[k], Ps[k]
            kl_all[k] = kl_gaussian(m0, P0, m, P)
        iensf_kl_rep[f'iensf_{N_iter}_rep{rep_id}'] = kl_all

N_iter_select = 5
kl_all_method['iensf'] = np.array([iensf_kl_rep[f'iensf_{N_iter_select}_rep{rep_id}'] for rep_id in range(n_rep)])

In [ ]:
# plot kl 

name_map = {'pf':'PF', 'ensf':'EnSF', 'enkf':'EnKF', 'iensf':'IEnSF', 'kf':'KF'}
color_map = {'pf':'green', 'ensf':'brown', 'enkf':'blue', 'iensf':'red', 'kf':'orange'}
line_map = {'pf':':^', 'ensf':'--', 'enkf':':.', 'iensf':'-*', 'kf':'-'}
line_alpha = 0.7

qs = [0.1, 0.5, 0.90]
plt.figure(figsize=(8,5))
for name, kl_rep_all in kl_all_method.items():
    print(f"Method: {name}")
     # compute quantiles
    quantiles = np.quantile(kl_rep_all, qs, axis=0)   # shape (len(qs), 101)
    q_lo, q_mid, q_hi = quantiles  # unpack

    # plot all KL curves
    plt.plot(t, q_mid, line_map[name], color=color_map[name],  label=name_map[name], alpha=line_alpha)
    plt.fill_between(t, q_lo, q_hi, color=color_map[name], alpha=0.1)




plt.xlabel("time step")
plt.ylabel("KL divergence")
# plt.ylim(0.,2)
plt.yscale("log")
plt.legend()
plt.grid(which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.title("KL divergence to KF posterior")

plt.savefig('linear_DA_KL.png',dpi=200, bbox_inches="tight")
plt.savefig('linear_DA_KL.pdf',dpi=200, bbox_inches="tight")
plt.show()

In [ ]:

# get all kl for different N_iter
kl_iter = {}
for N_iter in N_iter_vec:
    kl_iter[N_iter] = np.array([iensf_kl_rep[f'iensf_{N_iter}_rep{rep_id}'] for rep_id in range(n_rep)])


qs = [0.1, 0.5, 0.90]
plt.figure(figsize=(8,5))
for N_iter, kl_rep_all in kl_iter.items():
    print(f"N_iter: {N_iter}")
     # compute quantiles
    quantiles = np.quantile(kl_rep_all, qs, axis=0)   # shape (len(qs), 101)
    q_lo, q_mid, q_hi = quantiles  # unpack

    # # plot all KL curves
    plt.plot(t, q_mid, alpha=0.8, label=f"Iter {N_iter}")
    # plt.fill_between(t, q_lo, q_hi, alpha=0.2)
    
plt.xlabel("time step")
plt.ylabel("KL divergence")
plt.ylim(0.,0.6)
# plt.yscale("log")
plt.legend()
plt.grid(which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.title("KL divergence VS. number of iterations")

plt.savefig('linear_DA_iter_t.png',dpi=200, bbox_inches="tight")
plt.savefig('linear_DA_iter_t.pdf',dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# KL divergece vs interation

# get all kl for different N_iter
kl_iter = {}
for N_iter in N_iter_vec:
    kl_iter[N_iter] = np.array([iensf_kl_rep[f'iensf_{N_iter}_rep{rep_id}'] for rep_id in range(n_rep)])


qs = [0.1, 0.5, 0.90]

mean_vec = []
q_lo_vec = []
q_hi_vec = []
for N_iter, kl_rep_all in kl_iter.items():
    print(f"N_iter: {N_iter}")
     # compute quantiles
    quantiles = np.quantile(kl_rep_all, qs)   # shape (len(qs), 101)
    q_lo, q_mid, q_hi = quantiles  # unpack
    kl_mean = np.mean(kl_rep_all)
    
    mean_vec.append(kl_mean)
    q_lo_vec.append(q_lo)
    q_hi_vec.append(q_hi)


kl_enkf = kl_all_method['enkf']
kl_pf = kl_all_method['pf']
kl_mean_pf = np.mean(kl_pf)
kl_mean_enkf = np.mean(kl_enkf)

q_lo_pf,_,q_hi_pf = np.quantile(kl_pf, [0.1, 0.5, 0.90])
q_lo_enkf,_,q_hi_enkf = np.quantile(kl_enkf, [0.1, 0.5, 0.90])


iter_index=np.arange(5)
plt.figure(figsize=(8,5))
# # plot all KL curves
plt.plot(iter_index, mean_vec, '.-', alpha=0.8, label=f"IEnSF Mean")
plt.fill_between(iter_index, q_lo_vec, q_hi_vec, alpha=0.2, label='0.1–0.9 quantile')

# plt.plot(iter_index, iter_index*0 + kl_mean_pf, label=f"PF")
# plt.fill_between(iter_index, q_lo_pf, q_hi_pf, alpha=0.2)

# plt.plot(iter_index, iter_index*0 + kl_mean_enkf, label=f"EnKF")
# plt.fill_between(iter_index, q_lo_enkf, q_hi_enkf, alpha=0.2)


plt.xlabel("Number of iterations")
plt.ylabel("KL divergence")
plt.ylim(0.,0.5)
# plt.yscale("log")
plt.legend()
plt.xticks(iter_index) 
plt.grid(which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.title("KL divergence VS. Number of iterations")

plt.savefig('linear_DA_iter.png',dpi=200, bbox_inches="tight")
plt.savefig('linear_DA_iter.pdf',dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# add the iensf iter 5 to data list 
N_iter_select = 5 
iensf_iter5 = []
for rep_id in range(n_rep):
    result_temp = iensf_results[f'iensf_{N_iter_select}_rep{rep_id}']
    ms, Ps = result_temp
    iensf_iter5.append((ms, Ps))
load_result_all['iensf'] = iensf_iter5

In [ ]:
# plot estimation

plot_order = ['iensf', 'ensf', 'enkf', 'pf', 'kf']

# get est and spread
ests = {}
spreads = {}

# kf
ests['kf'] = ms_kf
spreads['kf'] = np.sqrt(np.diagonal(Ps_kf, axis1=1, axis2=2))  # (T,n)

# get est and spread for other methods
for method_name in plot_order:
    if method_name == 'kf':
        continue
    loaded_results = load_result_all[method_name]
    est = []
    spread = []
    for rep_id, (ms, Ps) in enumerate(loaded_results):
        est.append(ms)  # (T,n)
        spread.append(np.sqrt(np.diagonal(Ps, axis1=1, axis2=2))) # (T,n)
    ests[method_name] = np.array(est)
    spreads[method_name] = np.array(spread)  # (n_rep, T, n)
        
        


In [ ]:
# name_map = {'pf':'PF', 'ensf':'EnSF', 'enkf':'EnKF', 'iensf':'IEnSF', 'kf':'KF'}
# color_map = {'pf':'orange', 'ensf':'brown', 'enkf':'blue', 'iensf':'red', 'kf':'green'}
# line_map = {'pf':':^', 'ensf':'--', 'enkf':':.', 'iensf':'-*', 'kf':'-'}
line_alpha = 0.7

# plot 
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# state
ax_f = axes[:,0]
for i, ax in enumerate(ax_f):
    for name in plot_order:
        data = ests[name][...,i]
        if len(data.shape) == 1:
            # KF single run, others multiple runs
            # plot
            ax.plot(data, line_map[name], color=color_map[name], label=name_map[name], alpha=line_alpha)  #
            
        else:
            est_mean = np.mean(data, axis=0)
            qs = [0.1,  0.90]
            quantiles = np.quantile(data, qs, axis=0)   # shape (len(qs), 101)
            q_lo, q_hi = quantiles  # unpack
            # plot
            ax.plot(est_mean, line_map[name], color=color_map[name],  label=name_map[name], alpha=line_alpha)  #
            # ax.fill_between(np.arange(len(q_lo)), q_lo, q_hi,color=color_map[name], alpha=0.1)
            
    ax.plot(state_true[:,i],'-' ,color='black', label='true')
    if i==0:
        ax.plot(y, '*', color='purple', label='obs', alpha=0.6)
    ax.grid(which='both')
    ax.set_title(f'State (Dim {i+1})')
    # ax.legend()

# spread 
ax_f = axes[:,1]
for i, ax in enumerate(ax_f):
    for name in plot_order:
        data = spreads[name][...,i]
        if len(data.shape) == 1:
            # KF single run, others multiple runs
            ax.plot(data, line_map[name], color=color_map[name], label=name_map[name], alpha=line_alpha)  #
        else:
            spread_mean = np.mean(data, axis=0)
            qs = [0.1,  0.90]
            quantiles = np.quantile(data, qs, axis=0)   # shape (len(qs), 101)
            q_lo, q_hi = quantiles  # unpack
            # plot 
            ax.plot(spread_mean, line_map[name], color=color_map[name],  label=name_map[name], alpha=line_alpha)
            # ax.fill_between(np.arange(len(q_lo)), q_lo, q_hi,color=color_map[name], alpha=0.1)

    ax.grid(which='both')
    ax.set_title(f'Spread (Dim {i+1})')
    # ax.set_yscale('log')
    # ax.legend()
    

axes[0,1].set_ylim(0.,1.1)
# Get handles/labels only from ax2
handles, labels = axes[0,0].get_legend_handles_labels()

# Put single legend at bottom for ax2 only
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=4,
    fontsize=12
)

plt.tight_layout()
# plt.suptitle("Linear DA: State Estimation and Spread", fontsize=16)

plt.savefig('linear_DA_est_spread.png',dpi=200, bbox_inches="tight")
plt.savefig('linear_DA_est_spread.pdf',dpi=200, bbox_inches="tight")
plt.show()
